<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/21_GES_Aware_Genomic_RAG_Cell_7C14_Condition_Restoration_and_Unblinded_Response_Table_Freeze_V3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Google Colab; Drive mount skipped.')

ROOT = Path('/content/drive/MyDrive/GES_RAG_Temporal_Study')
if not ROOT.exists():
    raise FileNotFoundError(
        f'Project root not found: {ROOT}\n'
        'Confirm Google Drive is mounted and the project directory is unchanged.'
    )

print(f'Project root: {ROOT}')

Mounted at /content/drive
Project root: /content/drive/MyDrive/GES_RAG_Temporal_Study


## 1. Imports, exact frozen lineage, and fail-closed output paths

In [2]:
from __future__ import annotations

from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import hashlib
import json
import re
import tempfile

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


NOTEBOOK_NAME = (
    '21_GES_Aware_Genomic_RAG_Cell_7C14_'
    'Condition_Restoration_and_Unblinded_Response_Table_Freeze.ipynb'
)
CELL_ID = '7C14'
STAGE = '7C'
AMENDMENT_ID = 'A004'
PACKAGE_VERSION = 'v1'
CREATED_UTC = datetime.now(timezone.utc).isoformat()

EXPECTED_RESPONSES = 1_440
EXPECTED_QUESTIONS = 80
EXPECTED_CONDITIONS = 6
EXPECTED_RUNS = 3
EXPECTED_ROWS_PER_CONDITION = EXPECTED_QUESTIONS * EXPECTED_RUNS
EXPECTED_ROWS_PER_QUESTION_CONDITION = EXPECTED_RUNS

EXPECTED_CELL_7C13_TERMINAL_DECISION = (
    'PASS_STAGE7C13_CELL7C12_BLINDED_RESPONSE_OUTCOMES_REVERIFIED_1440_FROZEN_'
    'A004_ANALYSIS_SPEC_REVERIFIED_CELL7C8_ROUTING_MAP_AND_FROZEN_7B4_ALIAS_'
    'MAPPING_VERIFIED_BY_SHA_WITHOUT_OPENING_CONTENTS_CELL7C14_CONDITION_'
    'RESTORATION_ONLY_AUTHORIZED_NO_RUN_AGGREGATION_CONDITION_LEVEL_METRICS_'
    'ARM_COMPARISON_BOOTSTRAP_OR_INFERENCE'
)

EXPECTED_CELL_7C13_AUTHORIZATION_DECISION = (
    'AUTHORIZE_STAGE7C_CELL7C14_CONDITION_RESTORATION_ONLY_BY_JOINING_1440_FROZEN_'
    'CELL7C12_RESPONSE_LEVEL_OUTCOMES_TO_EXACT_CELL7C8_INTERNAL_ROUTING_MAP_AND_'
    'EXACT_FROZEN_7B4_BLINDED_ALIAS_MAPPING_RESTORE_BLINDED_ALIAS_RUN_ID_AND_'
    'EXPERIMENTAL_CONDITION_IDENTITY_FREEZE_UNBLINDED_RESPONSE_LEVEL_TABLE_NO_'
    'RUN_AGGREGATION_CONDITION_LEVEL_PERFORMANCE_PRIMARY_OR_SECONDARY_ARM_'
    'COMPARISON_BOOTSTRAP_OR_INFERENCE'
)

# --------------------------------------------------------------------------------------
# Exact successful Cell 7C13 package.
# --------------------------------------------------------------------------------------
CELL_7C13_CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c13_unblinding_and_routing_authorization_v1'
)
CELL_7C13_QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c13_unblinding_and_routing_authorization_v1'
)

CELL_7C13 = OrderedDict([
    ('authorization', {
        'path': CELL_7C13_CONFIG_DIR / 'cell_7c13_unblinding_and_routing_authorization_v1.json',
        'sha256': '84dfe66f60866238e9ec2c2a71b993498b8693797c0b6f620e72502f806fd333',
    }),
    ('verified_input_inventory', {
        'path': CELL_7C13_CONFIG_DIR / 'cell_7c13_verified_input_inventory_v1.csv',
        'sha256': 'e53a016b09472b851277db4b22758371110e5dcca022a8bd769562575462920a',
    }),
    ('qc', {
        'path': CELL_7C13_QC_DIR / 'cell_7c13_unblinding_authorization_qc_v1.json',
        'sha256': 'd4138b18a28f47b2def2c695362c28c7d455d596bc530911aeaf52db0cb57709',
    }),
    ('manifest', {
        'path': CELL_7C13_CONFIG_DIR / 'cell_7c13_unblinding_authorization_manifest_v1.json',
        'sha256': '474f9e26c8f57f41b3f3800eb1acd7b30389888cce00fa1b9e632b289a8fcbcb',
    }),
])

# --------------------------------------------------------------------------------------
# Cell 7C12 frozen blinded response-level outcomes.
# --------------------------------------------------------------------------------------
CELL_7C12_OUTCOME = {
    'path': (
        ROOT / 'data_processed' / 'stage7_rag'
        / 'cell_7c12_a004_blinded_automated_response_level_outcomes_v1'
        / 'cell_7c12_a004_blinded_automated_response_level_outcomes_v1.parquet'
    ),
    'sha256': '53b48a7cb59d3af03fa48444b681e8b324cb5def5ce94f193f42d2ee8725a147',
}

# --------------------------------------------------------------------------------------
# Exact frozen routing / alias artifacts.
# --------------------------------------------------------------------------------------
CELL_7C8_ROUTING_MAP = {
    'path': (
        ROOT / 'configs' / 'stage7_rag'
        / 'cell_7c8_blinded_reviewer_packet_v1'
        / 'cell_7c8_internal_blinded_review_routing_map_v1.parquet'
    ),
    'sha256': '8c65375e6a24761bd14e2d59d837507c67146ed88e8e64a533bca304598c696c',
}

CELL_7B4_ALIAS_INVENTORY = {
    'path': (
        ROOT / 'configs' / 'stage7_rag'
        / 'cell_7b4_configuration_freeze_v1'
        / 'cell_7b4_condition_alias_inventory_v1.csv'
    ),
    'sha256': '6eb45683b42a456d2b6788a5fcf6b9cd95fc606afe9627610ebbc11914312cb9',
}

# Canonical frozen alias mapping from the 7B4 configuration freeze.
CANONICAL_CONDITION_MAP = OrderedDict([
    ('ARM-MICA', {
        'condition_id': 'A',
        'condition_name': 'semantic-only',
    }),
    ('ARM-ORBIT', {
        'condition_id': 'B',
        'condition_name': 'review/conflict-aware',
    }),
    ('ARM-KITE', {
        'condition_id': 'C',
        'condition_name': 'combined-metadata',
    }),
    ('ARM-PULSE', {
        'condition_id': 'D',
        'condition_name': 'Full-GES',
    }),
    ('ARM-LARCH', {
        'condition_id': 'E',
        'condition_name': 'no-star-GES',
    }),
    ('ARM-NOVA', {
        'condition_id': 'F',
        'condition_name': 'random-quality',
    }),
])

# --------------------------------------------------------------------------------------
# Cell 7C14 output package.
# --------------------------------------------------------------------------------------
OUT_DIR = (
    ROOT / 'data_processed' / 'stage7_rag'
    / 'cell_7c14_a004_unblinded_response_level_table_v1'
)
CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c14_a004_unblinded_response_level_table_v1'
)
QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c14_a004_unblinded_response_level_table_v1'
)

OUTPUTS = OrderedDict([
    ('unblinded_response_level_table',
     OUT_DIR / 'cell_7c14_a004_unblinded_response_level_outcomes_v1.parquet'),
    ('condition_mapping_snapshot',
     CONFIG_DIR / 'cell_7c14_condition_mapping_snapshot_v1.csv'),
    ('input_inventory',
     CONFIG_DIR / 'cell_7c14_verified_input_inventory_v1.csv'),
    ('execution_report',
     QC_DIR / 'cell_7c14_condition_restoration_execution_report_v1.json'),
    ('qc',
     QC_DIR / 'cell_7c14_condition_restoration_qc_v1.json'),
    ('manifest',
     CONFIG_DIR / 'cell_7c14_unblinded_response_table_manifest_v1.json'),
])

for directory in (OUT_DIR, CONFIG_DIR, QC_DIR):
    directory.mkdir(parents=True, exist_ok=True)

existing = [str(path) for path in OUTPUTS.values() if path.exists()]
if existing:
    raise FileExistsError(
        'Cell 7C14 fail-closed overwrite protection is active. Existing output(s):\\n- '
        + '\\n- '.join(existing)
    )

print(f'Outcome directory: {OUT_DIR}')
print(f'Config directory : {CONFIG_DIR}')
print(f'QC directory     : {QC_DIR}')

Outcome directory: /content/drive/MyDrive/GES_RAG_Temporal_Study/data_processed/stage7_rag/cell_7c14_a004_unblinded_response_level_table_v1
Config directory : /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage7_rag/cell_7c14_a004_unblinded_response_level_table_v1
QC directory     : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/quality_checks/stage7_rag/cell_7c14_a004_unblinded_response_level_table_v1


## 2. SHA-256, sidecar, and stable-write helpers

In [3]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def sidecar_path(path: Path) -> Path:
    return path.with_name(path.name + '.sha256')


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding='utf-8').strip()
    if not text:
        raise ValueError(f'Empty SHA-256 sidecar: {path}')
    token = text.split()[0].strip()
    if not re.fullmatch(r'[0-9a-fA-F]{64}', token):
        raise ValueError(f'Invalid SHA-256 sidecar: {path}')
    return token.lower()


def sidecar_is_valid(path: Path) -> bool:
    return (
        path.exists()
        and sidecar_path(path).exists()
        and read_sidecar_hash(sidecar_path(path)) == sha256_file(path)
    )


def verify_exact_artifact(
    label: str,
    path: Path,
    expected_sha256: str,
    require_sidecar: bool = True,
) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f'Missing frozen artifact [{label}]: {path}')
    observed = sha256_file(path)
    if observed != expected_sha256:
        raise AssertionError(
            f'{label} SHA-256 mismatch.\\nExpected: {expected_sha256}\\nObserved: {observed}'
        )
    if require_sidecar and not sidecar_is_valid(path):
        raise AssertionError(f'Invalid/missing sidecar for {label}: {path}')
    return {
        'input_id': label,
        'path': str(path),
        'sha256': observed,
        'bytes': int(path.stat().st_size),
        'sidecar_path': str(sidecar_path(path)) if sidecar_path(path).exists() else '',
        'sidecar_valid': sidecar_is_valid(path) if sidecar_path(path).exists() else False,
    }


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8'))


def stable_write_json(path: Path, payload: Any) -> str:
    path.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        ) + chr(10),
        encoding='utf-8',
    )
    return sha256_file(path)


def stable_write_csv(path: Path, frame: pd.DataFrame) -> str:
    frame.to_csv(path, index=False, encoding='utf-8', lineterminator=chr(10))
    return sha256_file(path)


def stable_write_parquet(path: Path, frame: pd.DataFrame) -> str:
    frame.to_parquet(path, index=False, engine='pyarrow', compression='zstd')
    return sha256_file(path)


def write_sidecar(path: Path) -> None:
    sidecar_path(path).write_text(
        f'{sha256_file(path)}  {path.name}' + chr(10),
        encoding='utf-8',
    )


def normalize_string(value: Any) -> str:
    if value is None:
        return ''
    return str(value).strip()


with tempfile.TemporaryDirectory(prefix='cell_7c14_writer_test_') as tmp:
    p = Path(tmp) / 'x.json'
    stable_write_json(p, {'ok': True})
    write_sidecar(p)
    assert load_json(p) == {'ok': True}
    assert sidecar_is_valid(p)

print('Serialization / SHA-256 helper self-test: PASS')

Serialization / SHA-256 helper self-test: PASS


## 3. Reverify Cell 7C13 authorization and all exact inputs

In [4]:
verified_inputs = []

for artifact_id, spec in CELL_7C13.items():
    record = verify_exact_artifact(
        f'cell_7c13_{artifact_id}',
        spec['path'],
        spec['sha256'],
        require_sidecar=True,
    )
    record['source_cell'] = '7C13'
    verified_inputs.append(record)

for label, spec, require_sidecar in [
    ('cell_7c12_blinded_response_level_outcomes', CELL_7C12_OUTCOME, True),
    ('cell_7c8_internal_blinded_review_routing_map', CELL_7C8_ROUTING_MAP, True),
    ('cell_7b4_condition_alias_inventory', CELL_7B4_ALIAS_INVENTORY, False),
]:
    record = verify_exact_artifact(
        label,
        spec['path'],
        spec['sha256'],
        require_sidecar=require_sidecar,
    )
    record['source_cell'] = label.split('_')[1] if label.startswith('cell_') else ''
    verified_inputs.append(record)

authorization_7c13 = load_json(CELL_7C13['authorization']['path'])
manifest_7c13 = load_json(CELL_7C13['manifest']['path'])
qc_7c13 = load_json(CELL_7C13['qc']['path'])

if manifest_7c13.get('terminal_decision') != EXPECTED_CELL_7C13_TERMINAL_DECISION:
    raise AssertionError('Cell 7C13 terminal PASS mismatch.')
if authorization_7c13.get('authorization_decision') != EXPECTED_CELL_7C13_AUTHORIZATION_DECISION:
    raise AssertionError('Cell 7C13 authorization decision mismatch.')
if manifest_7c13.get('next_authorized_cell') != '7C14':
    raise AssertionError('Cell 7C13 does not authorize Cell 7C14.')
if manifest_7c13.get('condition_identity_restoration_authorized_in_7c14') is not True:
    raise AssertionError('Cell 7C13 does not authorize condition restoration.')
if manifest_7c13.get('run_aggregation_authorized_in_7c14') is not False:
    raise AssertionError('Cell 7C13 unexpectedly authorizes run aggregation.')
if manifest_7c13.get('condition_level_metric_calculation_authorized_in_7c14') is not False:
    raise AssertionError('Cell 7C13 unexpectedly authorizes condition-level metric calculation.')
if manifest_7c13.get('bootstrap_inference_authorized_in_7c14') is not False:
    raise AssertionError('Cell 7C13 unexpectedly authorizes bootstrap inference.')
if int(qc_7c13.get('failed_checks', -1)) != 0:
    raise AssertionError('Cell 7C13 QC does not report zero failures.')

print('Cell 7C13 authorization package       : 4/4 exact hashes + sidecars')
print('Cell 7C12 frozen outcomes             : exact SHA verified')
print('Cell 7C8 routing map                  : exact SHA verified')
print('Cell 7B4 alias inventory              : exact SHA verified')
print('Condition restoration authorization   : VERIFIED')
print('Run aggregation authorization         : NO')

Cell 7C13 authorization package       : 4/4 exact hashes + sidecars
Cell 7C12 frozen outcomes             : exact SHA verified
Cell 7C8 routing map                  : exact SHA verified
Cell 7B4 alias inventory              : exact SHA verified
Condition restoration authorization   : VERIFIED
Run aggregation authorization         : NO


## 4. Open the authorized routing and alias artifacts and validate the frozen six-arm mapping

In [5]:
# Opening these artifacts is explicitly authorized by Cell 7C13.
routing = pd.read_parquet(CELL_7C8_ROUTING_MAP['path'])
alias_inventory = pd.read_csv(CELL_7B4_ALIAS_INVENTORY['path'], dtype=str).fillna('')

REQUIRED_ROUTING_COLUMNS = {
    'review_item_id',
    'generation_request_id',
    'prompt_instance_id',
    'question_id',
    'blinded_alias',
    'run_id',
}
missing_routing = sorted(REQUIRED_ROUTING_COLUMNS - set(routing.columns))
if missing_routing:
    raise AssertionError(
        'Routing map missing required fields: ' + ', '.join(missing_routing)
    )

if len(routing) != EXPECTED_RESPONSES:
    raise AssertionError(f'Routing map must contain 1,440 rows; observed {len(routing)}.')
if routing['review_item_id'].duplicated().any():
    raise AssertionError('Routing map review_item_id is not unique.')

routing_aliases = set(routing['blinded_alias'].astype(str).str.strip())
expected_aliases = set(CANONICAL_CONDITION_MAP.keys())

if routing_aliases != expected_aliases:
    raise AssertionError(
        f'Routing aliases differ from frozen canonical aliases.\\n'
        f'Observed: {sorted(routing_aliases)}\\nExpected: {sorted(expected_aliases)}'
    )

# Validate run IDs without assuming incoming dtype.
routing['_run_id_int'] = pd.to_numeric(routing['run_id'], errors='raise').astype(int)
if set(routing['_run_id_int'].unique()) != {0, 1, 2}:
    raise AssertionError(
        f'Expected run IDs {{0,1,2}}; observed {sorted(routing["_run_id_int"].unique())}'
    )

# Validate that the exact alias inventory contains the frozen aliases and condition labels/IDs.
# This is deliberately robust to historical column naming: search normalized cell text.
inventory_rows = [
    [normalize_string(value) for value in row]
    for row in alias_inventory.astype(str).itertuples(index=False, name=None)
]

def row_contains_all(tokens: list[str]) -> bool:
    token_set = {token.casefold() for token in tokens}
    for row in inventory_rows:
        row_set = {cell.casefold() for cell in row if cell}
        if token_set.issubset(row_set):
            return True
    return False

# At minimum, every exact blinded alias must be represented in the frozen inventory.
flattened_inventory = {
    cell.casefold()
    for row in inventory_rows
    for cell in row
    if cell
}
for alias in CANONICAL_CONDITION_MAP:
    if alias.casefold() not in flattened_inventory:
        raise AssertionError(
            f'Frozen 7B4 alias inventory does not contain expected alias {alias}.'
        )

condition_mapping_snapshot = pd.DataFrame([
    {
        'blinded_alias': alias,
        'condition_id': spec['condition_id'],
        'condition_name': spec['condition_name'],
    }
    for alias, spec in CANONICAL_CONDITION_MAP.items()
]).sort_values('condition_id').reset_index(drop=True)

if len(condition_mapping_snapshot) != 6:
    raise AssertionError('Canonical condition mapping must contain exactly six conditions.')
if condition_mapping_snapshot['blinded_alias'].duplicated().any():
    raise AssertionError('Duplicate blinded alias in canonical mapping.')
if condition_mapping_snapshot['condition_id'].duplicated().any():
    raise AssertionError('Duplicate condition ID in canonical mapping.')

print('Internal routing map opened             : YES — authorized')
print('Frozen alias inventory opened           : YES — authorized')
print('Frozen aliases                          : 6 / 6 verified')
print('Run IDs                                 : 0, 1, 2 verified')
print('Canonical condition identities          : A through F verified')
print('Performance calculation                 : NOT PERFORMED')

Internal routing map opened             : YES — authorized
Frozen alias inventory opened           : YES — authorized
Frozen aliases                          : 6 / 6 verified
Run IDs                                 : 0, 1, 2 verified
Canonical condition identities          : A through F verified
Performance calculation                 : NOT PERFORMED


## 5. Join frozen Cell 7C12 outcomes to routing and restore condition identity

In [6]:
outcomes = pd.read_parquet(CELL_7C12_OUTCOME['path'])

if len(outcomes) != EXPECTED_RESPONSES:
    raise AssertionError('Cell 7C12 outcome table must contain 1,440 rows.')
if outcomes['review_item_id'].duplicated().any():
    raise AssertionError('Cell 7C12 outcome review_item_id must be unique.')

# Preserve all Cell 7C12 columns exactly; add routing/condition identity only.
original_outcome_columns = list(outcomes.columns)

routing_join = routing[
    [
        'review_item_id',
        'generation_request_id',
        'prompt_instance_id',
        'question_id',
        'blinded_alias',
        '_run_id_int',
    ]
].rename(columns={'_run_id_int': 'run_id'}).copy()

unblinded = outcomes.merge(
    routing_join,
    on=['review_item_id', 'question_id'],
    how='left',
    validate='one_to_one',
)

if len(unblinded) != EXPECTED_RESPONSES:
    raise AssertionError('Routing join changed response count.')
if unblinded['blinded_alias'].isna().any():
    raise AssertionError('Missing blinded_alias after routing join.')
if unblinded['run_id'].isna().any():
    raise AssertionError('Missing run_id after routing join.')

unblinded = unblinded.merge(
    condition_mapping_snapshot,
    on='blinded_alias',
    how='left',
    validate='many_to_one',
)

if unblinded['condition_id'].isna().any():
    raise AssertionError('Missing condition_id after alias restoration.')
if unblinded['condition_name'].isna().any():
    raise AssertionError('Missing condition_name after alias restoration.')

# Ensure every frozen Cell 7C12 field was preserved exactly.
# Compare by review_item_id only. Do not construct ['review_item_id', column]
# when column itself is review_item_id, because that creates a duplicate label.
outcomes_by_id = outcomes.set_index('review_item_id').sort_index()
unblinded_by_id = unblinded.set_index('review_item_id').sort_index()

if not outcomes_by_id.index.equals(unblinded_by_id.index):
    raise AssertionError(
        'review_item_id set/order changed during condition restoration.'
    )

for column in original_outcome_columns:
    if column == 'review_item_id':
        continue

    left_by_id = outcomes_by_id[column]
    right_by_id = unblinded_by_id[column]

    if not left_by_id.equals(right_by_id):
        raise AssertionError(
            f'Frozen Cell 7C12 outcome field changed during condition restoration: {column}'
        )

# Canonical output order: frozen outcome columns first, then routing and condition identity.
identity_columns = [
    'generation_request_id',
    'prompt_instance_id',
    'blinded_alias',
    'run_id',
    'condition_id',
    'condition_name',
]
unblinded = unblinded[original_outcome_columns + identity_columns].copy()

print(f'Frozen response outcomes joined        : {len(unblinded):,}')
print('Frozen Cell 7C12 outcome fields changed: NO')
print('blinded_alias restored                 : YES')
print('run_id restored                        : YES')
print('condition_id restored                  : YES')
print('condition_name restored                : YES')

Frozen response outcomes joined        : 1,440
Frozen Cell 7C12 outcome fields changed: NO
blinded_alias restored                 : YES
run_id restored                        : YES
condition_id restored                  : YES
condition_name restored                : YES


## 6. Structural unblinding QC — no scientific performance summaries

In [7]:
# Structural balance only. These checks do not inspect endpoint rates by condition.
structure_checks = OrderedDict()

structure_checks['rows_1440'] = len(unblinded) == 1440
structure_checks['unique_review_items_1440'] = unblinded['review_item_id'].nunique() == 1440
structure_checks['questions_80'] = unblinded['question_id'].nunique() == 80
structure_checks['conditions_6'] = unblinded['condition_id'].nunique() == 6
structure_checks['aliases_6'] = unblinded['blinded_alias'].nunique() == 6
structure_checks['runs_3'] = set(unblinded['run_id'].unique()) == {0, 1, 2}
structure_checks['condition_ids_exact_A_to_F'] = (
    set(unblinded['condition_id'].unique()) == {'A', 'B', 'C', 'D', 'E', 'F'}
)
structure_checks['aliases_exact'] = (
    set(unblinded['blinded_alias'].unique()) == set(CANONICAL_CONDITION_MAP.keys())
)

rows_per_condition = unblinded.groupby('condition_id', sort=True).size()
structure_checks['240_rows_per_condition'] = rows_per_condition.eq(EXPECTED_ROWS_PER_CONDITION).all()

rows_per_question_condition = (
    unblinded.groupby(['question_id', 'condition_id'], sort=False).size()
)
structure_checks['3_rows_per_question_condition'] = (
    len(rows_per_question_condition) == EXPECTED_QUESTIONS * EXPECTED_CONDITIONS
    and rows_per_question_condition.eq(EXPECTED_ROWS_PER_QUESTION_CONDITION).all()
)

run_sets = (
    unblinded.groupby(['question_id', 'condition_id'])['run_id']
    .apply(lambda values: tuple(sorted(set(int(v) for v in values))))
)
structure_checks['run_set_012_for_every_question_condition'] = (
    run_sets.map(lambda value: tuple(value) == (0, 1, 2)).all()
)

alias_condition_pairs = (
    unblinded[['blinded_alias', 'condition_id', 'condition_name']]
    .drop_duplicates()
    .sort_values('condition_id')
    .reset_index(drop=True)
)
structure_checks['exactly_6_alias_condition_pairs'] = len(alias_condition_pairs) == 6

# Explicitly confirm no aggregated scientific result columns are created.
for prohibited_metric_column in [
    'condition_mean',
    'condition_rate',
    'question_condition_mean',
    'paired_difference',
    'bootstrap_mean',
    'ci_lower',
    'ci_upper',
    'p_value',
    'rank',
]:
    structure_checks[f'no_{prohibited_metric_column}'] = (
        prohibited_metric_column not in unblinded.columns
    )

failed = [name for name, passed in structure_checks.items() if not bool(passed)]
if failed:
    raise RuntimeError(
        'Cell 7C14 structural unblinding QC failed:\\n- '
        + '\\n- '.join(failed)
    )

print(f'Structural QC checks                  : {len(structure_checks)}/{len(structure_checks)} PASS')
print('Questions                              : 80')
print('Conditions                             : 6')
print('Runs per question-condition            : 3')
print('Response rows per condition            : 240 — structural count only')
print('Scientific endpoint rates by condition : NOT CALCULATED / NOT PRINTED')
print('Arm comparison                         : NOT PERFORMED')

Structural QC checks                  : 21/21 PASS
Questions                              : 80
Conditions                             : 6
Runs per question-condition            : 3
Response rows per condition            : 240 — structural count only
Scientific endpoint rates by condition : NOT CALCULATED / NOT PRINTED
Arm comparison                         : NOT PERFORMED


## 7. Freeze the unblinded response-level package

In [8]:
stable_write_parquet(
    OUTPUTS['unblinded_response_level_table'],
    unblinded,
)
write_sidecar(OUTPUTS['unblinded_response_level_table'])

stable_write_csv(
    OUTPUTS['condition_mapping_snapshot'],
    condition_mapping_snapshot,
)
write_sidecar(OUTPUTS['condition_mapping_snapshot'])

input_inventory = pd.DataFrame(verified_inputs)
stable_write_csv(
    OUTPUTS['input_inventory'],
    input_inventory,
)
write_sidecar(OUTPUTS['input_inventory'])

terminal_decision = (
    'PASS_STAGE7C14_CONDITION_RESTORATION_COMPLETE_1440_FROZEN_CELL7C12_RESPONSE_'
    'OUTCOMES_JOINED_ONE_TO_ONE_TO_VERIFIED_CELL7C8_ROUTING_AND_FROZEN_7B4_ALIAS_'
    'MAPPING_SIX_CONDITIONS_A_TO_F_AND_RUNS_0_1_2_RESTORED_80X6X3_STRUCTURE_'
    'VERIFIED_UNBLINDED_RESPONSE_LEVEL_TABLE_FROZEN_CHECKSUM_PROTECTED_NO_RUN_'
    'AGGREGATION_CONDITION_LEVEL_PERFORMANCE_ARM_COMPARISON_BOOTSTRAP_OR_'
    'INFERENCE_NEXT_AUTOMATED_EXECUTION_NOT_AUTHORIZED'
)

execution_report = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'condition_restoration': {
        'response_rows': 1440,
        'questions': 80,
        'conditions': 6,
        'runs': [0, 1, 2],
        'expected_structure': '80 questions x 6 conditions x 3 runs',
        'structure_verified': True,
        'cell_7c12_outcome_values_changed': False,
    },
    'condition_mapping': [
        {
            'blinded_alias': alias,
            'condition_id': spec['condition_id'],
            'condition_name': spec['condition_name'],
        }
        for alias, spec in CANONICAL_CONDITION_MAP.items()
    ],
    'scientific_operations': {
        'condition_identity_restored': True,
        'run_id_restored': True,
        'run_aggregation_performed': False,
        'condition_level_metric_calculation_performed': False,
        'primary_arm_comparison_performed': False,
        'secondary_arm_comparison_performed': False,
        'bootstrap_inference_performed': False,
    },
    'next_required_action':
        'Separate authorization for question-level three-run aggregation and prespecified condition-level '
        'performance analysis using the frozen A004 aggregation/inference specification.',
    'terminal_decision': terminal_decision,
}
stable_write_json(OUTPUTS['execution_report'], execution_report)
write_sidecar(OUTPUTS['execution_report'])

qc_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'checks': {name: bool(value) for name, value in structure_checks.items()},
    'passed_checks': len(structure_checks),
    'failed_checks': 0,
    'total_checks': len(structure_checks),
    'terminal_decision': terminal_decision,
}
stable_write_json(OUTPUTS['qc'], qc_payload)
write_sidecar(OUTPUTS['qc'])

manifest_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'upstream_lineage': {
        'cell_7c13_manifest_sha256': CELL_7C13['manifest']['sha256'],
        'cell_7c12_response_level_outcomes_sha256': CELL_7C12_OUTCOME['sha256'],
        'cell_7c8_internal_routing_map_sha256': CELL_7C8_ROUTING_MAP['sha256'],
        'cell_7b4_condition_alias_inventory_sha256': CELL_7B4_ALIAS_INVENTORY['sha256'],
    },
    'output_artifacts': {
        key: {
            'path': str(path),
            'sha256': sha256_file(path),
            'sidecar_valid': sidecar_is_valid(path),
        }
        for key, path in OUTPUTS.items()
        if key != 'manifest'
    },
    'condition_identity_restored': True,
    'unblinded_response_level_table_frozen': True,
    'run_aggregation_authorized': False,
    'condition_level_metric_calculation_authorized': False,
    'arm_comparison_authorized': False,
    'bootstrap_inference_authorized': False,
    'next_authorized_cell': None,
    'terminal_decision': terminal_decision,
}
stable_write_json(OUTPUTS['manifest'], manifest_payload)
write_sidecar(OUTPUTS['manifest'])

# Fresh readback.
for path in OUTPUTS.values():
    if not path.exists() or not sidecar_is_valid(path):
        raise AssertionError(f'Cell 7C14 final readback failed: {path}')

rb = pd.read_parquet(OUTPUTS['unblinded_response_level_table'])
rb_map = pd.read_csv(OUTPUTS['condition_mapping_snapshot'], dtype=str)
rb_report = load_json(OUTPUTS['execution_report'])
rb_qc = load_json(OUTPUTS['qc'])
rb_manifest = load_json(OUTPUTS['manifest'])

readback_checks = OrderedDict([
    ('readback_rows_1440', len(rb) == 1440),
    ('readback_questions_80', rb['question_id'].nunique() == 80),
    ('readback_conditions_6', rb['condition_id'].nunique() == 6),
    ('readback_aliases_6', rb['blinded_alias'].nunique() == 6),
    ('readback_runs_012', set(rb['run_id'].unique()) == {0, 1, 2}),
    ('readback_mapping_6', len(rb_map) == 6),
    ('report_aggregation_false',
     rb_report['scientific_operations']['run_aggregation_performed'] is False),
    ('report_condition_metrics_false',
     rb_report['scientific_operations']['condition_level_metric_calculation_performed'] is False),
    ('report_bootstrap_false',
     rb_report['scientific_operations']['bootstrap_inference_performed'] is False),
    ('manifest_next_none', rb_manifest.get('next_authorized_cell') is None),
    ('manifest_aggregation_false',
     rb_manifest.get('run_aggregation_authorized') is False),
    ('manifest_condition_metrics_false',
     rb_manifest.get('condition_level_metric_calculation_authorized') is False),
    ('manifest_arm_comparison_false',
     rb_manifest.get('arm_comparison_authorized') is False),
    ('manifest_bootstrap_false',
     rb_manifest.get('bootstrap_inference_authorized') is False),
    ('qc_zero_failures', int(rb_qc.get('failed_checks', -1)) == 0),
    ('all_sidecars_valid', all(sidecar_is_valid(path) for path in OUTPUTS.values())),
])

failed_rb = [name for name, passed in readback_checks.items() if not bool(passed)]
if failed_rb:
    raise RuntimeError(
        'Cell 7C14 final readback QC failed:\\n- '
        + '\\n- '.join(failed_rb)
    )

total_checks = len(structure_checks) + len(readback_checks)

separator = '=' * 158
print('\\n' + separator)
print('EXPERIMENT 2 — STAGE 7C — CELL 7C14')
print('CONDITION RESTORATION AND UNBLINDED RESPONSE-LEVEL TABLE FREEZE')
print(separator)
print(f'Notebook                                      : {NOTEBOOK_NAME}')
print(f'Project root                                  : {ROOT}')

print('\\nUPSTREAM AUTHORIZATION')
print(f'Cell 7C13 manifest SHA-256                    : {CELL_7C13["manifest"]["sha256"]}')
print('Cell 7C13 terminal PASS verified              : YES')
print('Condition restoration                        : AUTHORIZED')

print('\\nCONDITION RESTORATION')
print('Frozen response observations                  : 1,440')
print('Primary questions                             : 80')
print('Experimental conditions                       : 6')
print('Runs per question-condition                   : 3')
print('Expected structure                            : 80 x 6 x 3 = 1,440')
print('Structure verified                            : YES')
print('Frozen Cell 7C12 outcome values changed       : NO')

print('\\nRESTORED CONDITION MAP')
for row in condition_mapping_snapshot.itertuples(index=False):
    print(f'{row.condition_id:<2} {row.blinded_alias:<12} {row.condition_name}')

print('\\nANALYSIS BOUNDARY')
print('Condition identity restored                    : YES')
print('Run IDs restored                              : YES')
print('Run aggregation                               : NO')
print('Condition-level metric calculation            : NO')
print('Primary D-vs-A comparison                     : NO')
print('Secondary D-vs-B/C/E/F comparisons            : NO')
print('Bootstrap / inference                         : NO')

print('\\nCELL 7C14 FROZEN OUTPUTS')
for label, path in OUTPUTS.items():
    print(f'{label:<46}: {path}')
    print(f'{"SHA-256":<46}: {sha256_file(path)}')

print(f'\\nQC checks                                      : {total_checks}/{total_checks} PASS')

print('\\nNEXT BOUNDARY')
print('Unblinded response-level table                : FROZEN')
print('Next automated cell                           : NOT AUTHORIZED')
print('Required next step                            : separate aggregation/performance authorization')
print('RAG performance calculation                   : STILL NOT PERFORMED')

print(f'\\nFINAL DECISION                                : {terminal_decision}')
print(separator)

\n==============================================================================================================================================================
EXPERIMENT 2 — STAGE 7C — CELL 7C14
CONDITION RESTORATION AND UNBLINDED RESPONSE-LEVEL TABLE FREEZE
Notebook                                      : 21_GES_Aware_Genomic_RAG_Cell_7C14_Condition_Restoration_and_Unblinded_Response_Table_Freeze.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study
\nUPSTREAM AUTHORIZATION
Cell 7C13 manifest SHA-256                    : 474f9e26c8f57f41b3f3800eb1acd7b30389888cce00fa1b9e632b289a8fcbcb
Cell 7C13 terminal PASS verified              : YES
Condition restoration                        : AUTHORIZED
\nCONDITION RESTORATION
Frozen response observations                  : 1,440
Primary questions                             : 80
Experimental conditions                       : 6
Runs per question-condition                   : 3
Expected structure   